In [1]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Load Function 3 data
X = np.load("function3/initial_inputs.npy")
Y = np.load("function3/initial_outputs.npy")
print("X shape:", X.shape)
print("Y shape:", Y.shape)


print(X)


print(Y)

X shape: (15, 3)
Y shape: (15,)
[[0.17152521 0.34391687 0.2487372 ]
 [0.24211446 0.64407427 0.27243281]
 [0.53490572 0.39850092 0.17338873]
 [0.49258141 0.61159319 0.34017639]
 [0.13462167 0.21991724 0.45820622]
 [0.34552327 0.94135983 0.26936348]
 [0.15183663 0.43999062 0.99088187]
 [0.64550284 0.39714294 0.91977134]
 [0.74691195 0.28419631 0.22629985]
 [0.17047699 0.6970324  0.14916943]
 [0.22054934 0.29782524 0.34355534]
 [0.66601366 0.67198515 0.2462953 ]
 [0.04680895 0.23136024 0.77061759]
 [0.60009728 0.72513573 0.06608864]
 [0.96599485 0.86111969 0.56682913]]
[-0.1121222  -0.08796286 -0.11141465 -0.03483531 -0.04800758 -0.11062091
 -0.39892551 -0.11386851 -0.13146061 -0.09418956 -0.04694741 -0.10596504
 -0.11804826 -0.03637783 -0.05675837]


In [2]:
kernel = C(1.0) * RBF(length_scale=0.2)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    random_state=42
)

gp.fit(X, Y)

rng = np.random.default_rng(42)
candidates = rng.uniform(0, 1, size=(10000, X.shape[1]))

mean, std = gp.predict(candidates, return_std=True)

kappa = 2.5
ucb = mean + kappa * std

best_ucb_index = np.argmax(ucb)
query = candidates[best_ucb_index]

print("Best current x:", X[np.argmax(Y)])
print("Best current y:", np.max(Y))
print("Suggested query:", query)
print("Portal format:", "-".join(f"{v:.6f}" for v in query))
print("Predicted mean:", mean[best_ucb_index])
print("Predicted std:", std[best_ucb_index])
print("UCB score:", ucb[best_ucb_index])

Best current x: [0.49258141 0.61159319 0.34017639]
Best current y: -0.034835313350078584
Suggested query: [0.43307236 0.32477111 0.53773065]
Portal format: 0.433072-0.324771-0.537731
Predicted mean: -0.03826754900229139
Predicted std: 0.07906491260093727
UCB score: 0.15939473250005182


In [3]:
import numpy as np
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
#For Function 3, I used Expected Improvement because Week 1 improved the best value. Since the goal is still maximisation, I treated values closer to zero as better. 
#The EI-selected point had a predicted mean close to zero and some uncertainty, so it was a good candidate for further improvement rather than just a manual local guess.
# Load original Function 3 data
X = np.load("function3/initial_inputs.npy")
Y = np.load("function3/initial_outputs.npy")

# Add Week 1 query and output
week1_x = np.array([[0.433072, 0.324771, 0.537731]])
week1_y = np.array([-0.022514823783463457])

X = np.vstack([X, week1_x])
Y = np.append(Y, week1_y)

print("Updated X shape:", X.shape)
print("Updated Y shape:", Y.shape)

print("Best current x:", X[np.argmax(Y)])
print("Best current y:", np.max(Y))

# Fit GP surrogate model
kernel = C(1.0) * RBF(length_scale=0.2)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    random_state=42
)

gp.fit(X, Y)

# Generate candidate points
rng = np.random.default_rng(42)
candidates = rng.uniform(0, 1, size=(10000, X.shape[1]))

# Predict mean and uncertainty
mean, std = gp.predict(candidates, return_std=True)

# Expected Improvement acquisition
y_best = np.max(Y)
std_safe = std + 1e-12

improvement = mean - y_best
z = improvement / std_safe
ei = improvement * norm.cdf(z) + std_safe * norm.pdf(z)

best_ei_index = np.argmax(ei)
query = candidates[best_ei_index]

print("Acquisition used: Expected Improvement")
print("Suggested query:", query)
print("Portal format:", "-".join(f"{v:.6f}" for v in query))
print("Predicted mean:", mean[best_ei_index])
print("Predicted std:", std[best_ei_index])
print("EI score:", ei[best_ei_index])

Updated X shape: (16, 3)
Updated Y shape: (16,)
Best current x: [0.433072 0.324771 0.537731]
Best current y: -0.022514823783463457
Acquisition used: Expected Improvement
Suggested query: [0.36435188 0.40431212 0.45182228]
Portal format: 0.364352-0.404312-0.451822
Predicted mean: -0.0017915538447339924
Predicted std: 0.03222277838183981
EI score: 0.025787168475077633


In [4]:
import numpy as np
from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C


# For Function 3, I upgraded from pure Expected Improvement to a hybrid EI-UCB acquisition.
# Week 2 improved over Week 1, moving closer to zero, which is better because this is still a maximisation task.
# However, the best observed value is still negative, so I kept more exploration in the acquisition function
# rather than relying only on exploitation around the current best point.


# -----------------------------
# Load original Function 3 data
# -----------------------------
X = np.load("function3/initial_inputs.npy")
Y = np.load("function3/initial_outputs.npy")


# -----------------------------
# Add Week 1 and Week 2 results
# -----------------------------
week1_x = np.array([[0.433072, 0.324771, 0.537731]])
week1_y = np.array([-0.022514823783463457])

week2_x = np.array([[0.364352, 0.404312, 0.451822]])
week2_y = np.array([-0.014995206499628672])

X = np.vstack([X, week1_x, week2_x])
Y = np.append(Y, [week1_y[0], week2_y[0]])


print("Updated X shape:", X.shape)
print("Updated Y shape:", Y.shape)

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("Best current x:", best_x)
print("Best current y:", best_y)


# -----------------------------
# Fit GP surrogate model
# -----------------------------
kernel = (
    C(1.0, (1e-3, 1e3))
    * Matern(length_scale=np.ones(X.shape[1]) * 0.2, length_scale_bounds=(1e-2, 1.0), nu=2.5)
    + WhiteKernel(noise_level=1e-8, noise_level_bounds=(1e-10, 1e-3))
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

gp.fit(X, Y)

print("Fitted kernel:", gp.kernel_)


# -----------------------------
# Generate candidate points
# -----------------------------
rng = np.random.default_rng(42)
dim = X.shape[1]

# Global candidates keep exploration open because Function 3 is not solved yet.
global_candidates = rng.uniform(0, 1, size=(20000, dim))

# Local candidates around current best point, since Week 2 improved.
local_candidates = rng.normal(loc=best_x, scale=0.10, size=(15000, dim))
local_candidates = np.clip(local_candidates, 0, 1)

candidates = np.vstack([global_candidates, local_candidates])


# -----------------------------
# Predict mean and uncertainty
# -----------------------------
mean, std = gp.predict(candidates, return_std=True)

y_best = np.max(Y)
std_safe = std + 1e-12


# -----------------------------
# Expected Improvement
# -----------------------------
improvement = mean - y_best
z = improvement / std_safe
ei = improvement * norm.cdf(z) + std_safe * norm.pdf(z)


# -----------------------------
# Upper Confidence Bound
# -----------------------------
# Slightly higher kappa than Function 2 because Function 3 still has weak/negative values.
kappa = 2.0
ucb = mean + kappa * std


# -----------------------------
# Hybrid EI-UCB acquisition
# -----------------------------
ei_norm = (ei - np.min(ei)) / (np.max(ei) - np.min(ei) + 1e-12)
ucb_norm = (ucb - np.min(ucb)) / (np.max(ucb) - np.min(ucb) + 1e-12)

# More balanced than Function 2 because Function 3 still needs exploration.
hybrid_score = 0.55 * ei_norm + 0.45 * ucb_norm

best_hybrid_index = np.argmax(hybrid_score)
query = candidates[best_hybrid_index]


# -----------------------------
# Output results
# -----------------------------
print("Acquisition used: Hybrid Expected Improvement + UCB")
print("Suggested query:", query)

print("Portal format with hyphens:")
print("-".join(f"{v:.6f}" for v in query))

print("Portal format with x labels:")
print(",".join(f"x{i+1}:{v:.6f}" for i, v in enumerate(query)))

print("Predicted mean:", mean[best_hybrid_index])
print("Predicted std:", std[best_hybrid_index])
print("EI score:", ei[best_hybrid_index])
print("UCB score:", ucb[best_hybrid_index])
print("Hybrid score:", hybrid_score[best_hybrid_index])

Updated X shape: (17, 3)
Updated Y shape: (17,)
Best current x: [0.364352 0.404312 0.451822]
Best current y: -0.014995206499628672
Fitted kernel: 1.38**2 * Matern(length_scale=[0.737, 1, 0.21], nu=2.5) + WhiteKernel(noise_level=2.11e-10)
Acquisition used: Hybrid Expected Improvement + UCB
Suggested query: [0.99729169 0.02760536 0.76223866]
Portal format with hyphens:
0.997292-0.027605-0.762239
Portal format with x labels:
x1:0.997292,x2:0.027605,x3:0.762239
Predicted mean: -0.019674818691563956
Predicted std: 0.08637785673750545
EI score: 0.03217053119691352
UCB score: 0.15308089478344694
Hybrid score: 0.9999999999820737


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [5]:
# Keep only candidates whose predicted mean is at least as good as the current best
mean_filter = mean >= y_best

# Fallback: if no candidates pass the strict filter, relax slightly
if np.sum(mean_filter) == 0:
    mean_filter = mean >= (y_best - 0.005)

filtered_candidates = candidates[mean_filter]
filtered_mean = mean[mean_filter]
filtered_std = std[mean_filter]
filtered_ei = ei[mean_filter]
filtered_ucb = ucb[mean_filter]

ei_norm = (filtered_ei - np.min(filtered_ei)) / (np.max(filtered_ei) - np.min(filtered_ei) + 1e-12)
ucb_norm = (filtered_ucb - np.min(filtered_ucb)) / (np.max(filtered_ucb) - np.min(filtered_ucb) + 1e-12)

hybrid_score = 0.65 * ei_norm + 0.35 * ucb_norm

best_index = np.argmax(hybrid_score)
query = filtered_candidates[best_index]

print("Acquisition used: Strict filtered Hybrid EI + UCB")
print("Number of candidates passing filter:", np.sum(mean_filter))
print("Suggested query:", query)

print("Portal format with hyphens:")
print("-".join(f"{v:.6f}" for v in query))

print("Portal format with x labels:")
print(",".join(f"x{i+1}:{v:.6f}" for i, v in enumerate(query)))

print("Predicted mean:", filtered_mean[best_index])
print("Predicted std:", filtered_std[best_index])
print("EI score:", filtered_ei[best_index])
print("UCB score:", filtered_ucb[best_index])
print("Hybrid score:", hybrid_score[best_index])

Acquisition used: Strict filtered Hybrid EI + UCB
Number of candidates passing filter: 2757
Suggested query: [0.98787072 0.20685236 0.79027527]
Portal format with hyphens:
0.987871-0.206852-0.790275
Portal format with x labels:
x1:0.987871,x2:0.206852,x3:0.790275
Predicted mean: -0.014206881078098574
Predicted std: 0.07780980440727259
EI score: 0.031437376655884426
UCB score: 0.1414127277364466
Hybrid score: 0.9997532913900946


In [6]:
# week 4 

week3_x = np.array([[0.987871, 0.206852, 0.790275]])
week3_y = np.array([-0.09932090801789271])
X = np.vstack([X, week1_x, week2_x, week3_x])
Y = np.append(Y, [week1_y[0], week2_y[0], week3_y[0]])

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("Current best x:", best_x)
print("Current best y:", best_y)


X shape: (20, 3)
Y shape: (20,)
Current best x: [0.364352 0.404312 0.451822]
Current best y: -0.014995206499628672


In [7]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel as C,
    Matern,
    WhiteKernel
)

kernel = (
    C(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.full(3, 0.2),
        length_scale_bounds=(1e-2, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-8,
        noise_level_bounds=(1e-10, 1e-3)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("Fitted kernel:", gp.kernel_)

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 20 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 18 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 28 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/

Fitted kernel: 1.25**2 * Matern(length_scale=[2, 1.51, 0.0722], nu=2.5) + WhiteKernel(noise_level=1e-10)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [8]:
trust_radius = np.array([0.15, 0.15, 0.15])

lower_bounds = np.maximum(0, best_x - trust_radius)
upper_bounds = np.minimum(1, best_x + trust_radius)

bounds = list(zip(lower_bounds, upper_bounds))

print("Lower bounds:", lower_bounds)
print("Upper bounds:", upper_bounds)

Lower bounds: [0.214352 0.254312 0.301822]
Upper bounds: [0.514352 0.554312 0.601822]


In [9]:
kappa = 0.75

def negative_ucb(point):
    point = np.asarray(point).reshape(1, -1)

    mean, std = gp.predict(point, return_std=True)
    ucb = mean[0] + kappa * std[0]

    return -ucb

In [10]:
from scipy.optimize import minimize

rng = np.random.default_rng(42)

starting_points = [
    best_x,
    week1_x[0],
    week2_x[0]
]

random_starts = rng.uniform(
    lower_bounds,
    upper_bounds,
    size=(40, 3)
)

starting_points.extend(random_starts)

results = []

for start in starting_points:
    start = np.clip(start, lower_bounds, upper_bounds)

    result = minimize(
        negative_ucb,
        x0=start,
        method="L-BFGS-B",
        bounds=bounds
    )

    if result.success:
        results.append(result)

if not results:
    raise RuntimeError("No successful L-BFGS-B runs")

best_result = min(results, key=lambda result: result.fun)
query = best_result.x

In [11]:
query_mean, query_std = gp.predict(
    query.reshape(1, -1),
    return_std=True
)

query_ucb = query_mean[0] + kappa * query_std[0]

nearest_distance = np.min(
    np.linalg.norm(X - query, axis=1)
)

print("Method: Trust-region GP-UCB with L-BFGS-B")
print("Suggested Week 4 query:", query)

print(
    "Portal format:",
    "-".join(f"{value:.6f}" for value in query)
)

print("Predicted mean:", query_mean[0])
print("Predicted std:", query_std[0])
print("UCB:", query_ucb)
print("Distance to nearest observation:", nearest_distance)

Method: Trust-region GP-UCB with L-BFGS-B
Suggested Week 4 query: [0.514352   0.554312   0.41274749]
Portal format: 0.514352-0.554312-0.412747
Predicted mean: 0.0006074483162983868
Predicted std: 0.04469112672446044
UCB: 0.03412579335964372
Distance to nearest observation: 0.09498240957756707


In [12]:
matern_kernel = gp.kernel_.k1.k2
lengthscales = np.asarray(matern_kernel.length_scale)

importance = 1 / lengthscales
importance = importance / importance.sum()

print("Lengthscales:", lengthscales)
print("Normalised GP importance:")
print("x1:", importance[0])
print("x2:", importance[1])
print("x3:", importance[2])

Lengthscales: [2.         1.51280208 0.07218755]
Normalised GP importance:
x1: 0.03330263176503771
x2: 0.0440277444960762
x3: 0.9226696237388861
